# 00 · Prueba de humo del entorno (mini-paso 1.6)

**Proyecto Steam Reviews · Big Data & Cloud Computing (Entrega 1)**

Una *prueba de humo* es la primera prueba que se hace al encender una máquina nueva: si no sale humo, lo básico funciona. Aquí comprobamos que **todas las piezas del entorno hablan entre sí** antes de usar los 8 GB del dataset:

| # | Qué se comprueba | Resultado esperado |
|---|---|---|
| 1 | Jupyter (el *driver*) se conecta al **spark-master** | Sale el identificador de la aplicación (`app-...`) |
| 2 | Los **executors** del spark-worker ejecutan Python, con la misma versión que el driver | `('spark-worker', '3.13.15')` |
| 3 | Spark **reparte un cálculo** y junta el resultado | `285` |
| 4 | Spark **escribe en HDFS** (Parquet) | La carpeta `/tmp/prueba_humo` aparece en HDFS |
| 5 | Spark **lee de HDFS** lo que escribió | `1000000` filas, de ellas `500000` pares |

**Cómo ejecutarlo:** celda a celda, con **Shift + Enter**, de arriba abajo. No ejecutes la última celda (la que apaga Spark) hasta haber mirado las webs de Spark que se indican al final.

In [1]:
# ============================================================
# 1) Parámetros de la prueba
# ============================================================
# Dirección del "jefe de obra" de Spark (el master). El nombre "spark-master"
# solo existe dentro de la red de Docker; funciona porque Jupyter está en ella.
SPARK_MASTER = "spark://spark-master:7077"

# Dirección de HDFS: el NameNode escucha en el puerto 9000.
HDFS = "hdfs://namenode:9000"

# Carpeta de HDFS donde escribe la prueba. Se sobrescribe en cada ejecución
# (modo "overwrite"), así que se puede repetir las veces que haga falta.
RUTA_PRUEBA = HDFS + "/tmp/prueba_humo"

In [2]:
# ============================================================
# 2) Arrancar Spark conectado al cluster (comprobación 1)
# ============================================================
import sys
import time
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    # Nombre de la aplicación: es el que verás en localhost:8080 y localhost:4040.
    .appName("00_prueba_humo")
    # Trabajar en el cluster (master + worker), NO en modo local dentro de Jupyter.
    .master(SPARK_MASTER)
    # Memoria del driver (este notebook) y de cada executor (en el spark-worker).
    # Caben en los topes del docker-compose.yml: Jupyter 2 GB y worker 2 GB para executors.
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    # Rutas sin "hdfs://..." se entienden como rutas de HDFS.
    .config("spark.hadoop.fs.defaultFS", HDFS)
    # MUY IMPORTANTE: Spark escribe en HDFS como "cliente", y un cliente de Hadoop,
    # si nadie le dice nada, pide 3 copias de cada bloque (valor por defecto).
    # Solo tenemos 2 DataNodes: sin esta línea, todo lo que escriba Spark quedaría
    # "sub-replicado" (pidiendo una tercera copia imposible).
    .config("spark.hadoop.dfs.replication", "2")
    # Fechas en UTC, para que la partición por "year" sea igual en cualquier PC
    # (decisión del 16 oct; ver el documento explicativo).
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

print("Versión de Spark          :", spark.version)
print("Versión de Python (driver):", sys.version.split()[0])
print("Master                    :", spark.sparkContext.master)
print("Identificador de la app   :", spark.sparkContext.applicationId)

Versión de Spark          : 4.2.0
Versión de Python (driver): 3.13.15
Master                    : spark://spark-master:7077
Identificador de la app   : app-20261018100458-0000


In [3]:
# ============================================================
# 3) ¿Los executors ejecutan Python, y con la misma versión? (comprobación 2)
# ============================================================
# Esta función NO se ejecuta en Jupyter: Spark la envía a los executors del
# spark-worker, que la ejecutan en su propio proceso de Python y nos devuelven
# en qué máquina (contenedor) se ejecutó y con qué versión de Python.
def donde_y_que_python(_):
    import socket
    import sys
    return (socket.gethostname(), sys.version.split()[0])

resultado = (
    spark.sparkContext
    .parallelize(range(4), 4)      # 4 números repartidos en 4 trozos (particiones) -> 4 tareas
    .map(donde_y_que_python)       # cada tarea pregunta "¿dónde estoy y qué Python tengo?"
    .distinct()                    # quitamos respuestas repetidas
    .collect()                     # traemos el resultado al driver (este notebook)
)
print("Executors (máquina, Python):", resultado)
print("¿Misma versión que el driver?:", all(v == sys.version.split()[0] for _, v in resultado))

Executors (máquina, Python): [('spark-worker', '3.13.15')]
¿Misma versión que el driver?: True


In [4]:
# ============================================================
# 4) Un cálculo repartido de verdad (comprobación 3)
# ============================================================
# Suma de los cuadrados de 0..9 = 0+1+4+9+16+25+36+49+64+81 = 285.
# Los números se reparten en 2 particiones; cada executor eleva al cuadrado
# los suyos (map) y Spark junta las sumas parciales (reduce).
suma = spark.sparkContext.parallelize(range(10), 2).map(lambda x: x * x).sum()
print("Suma de cuadrados de 0 a 9:", suma, "(esperado: 285)")

Suma de cuadrados de 0 a 9: 285 (esperado: 285)


In [5]:
# ============================================================
# 5) Escribir en HDFS (comprobación 4)
# ============================================================
# Una tabla de 1 millón de filas generada por Spark (sin leer ningún archivo):
#   numero = 0, 1, 2, ..., 999999
#   es_par = True si el número es par
df = spark.range(0, 1_000_000).withColumnRenamed("id", "numero")
df = df.withColumn("es_par", df.numero % 2 == 0)

inicio = time.time()
# "overwrite": si la carpeta ya existe (de una ejecución anterior), se reemplaza.
df.write.mode("overwrite").parquet(RUTA_PRUEBA)
print("Escrito en", RUTA_PRUEBA, "en", round(time.time() - inicio, 1), "s")

Escrito en hdfs://namenode:9000/tmp/prueba_humo en 3.5 s


In [6]:
# ============================================================
# 6) Leer de HDFS lo que acabamos de escribir (comprobación 5)
# ============================================================
leido = spark.read.parquet(RUTA_PRUEBA)
total = leido.count()
pares = leido.filter(leido.es_par).count()
print("Filas leídas desde HDFS:", total, "(esperado: 1000000)")
print("De ellas, pares        :", pares, "(esperado: 500000)")
print("¿Todo correcto?        :", total == 1_000_000 and pares == 500_000)

Filas leídas desde HDFS: 1000000 (esperado: 1000000)
De ellas, pares        : 500000 (esperado: 500000)
¿Todo correcto?        : True


## Antes de apagar: mirar las webs de Spark

Con Spark todavía encendido:

- **[http://localhost:8080](http://localhost:8080)** (spark-master): en *Running Applications* debe aparecer `00_prueba_humo`.
- **[http://localhost:4040](http://localhost:4040)** (la web de *esta* aplicación): pestaña *Jobs* (los trabajos que hemos lanzado) y pestaña *Executors* (el executor del spark-worker).

Y desde PowerShell, para ver lo escrito **desde el lado de HDFS**:

```powershell
docker exec namenode hdfs dfs -ls /tmp/prueba_humo
docker exec namenode hdfs fsck /tmp/prueba_humo -files -blocks -locations
```

Después, ejecuta la última celda para apagar Spark y liberar memoria.

In [7]:
# ============================================================
# 7) Apagar Spark (libera la memoria del driver y de los executors)
# ============================================================
spark.stop()
print("Spark apagado.")

Spark apagado.
